In [2]:
import pandas as pd
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error

In [4]:
import shap
import warnings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
print("All libraries loaded ✓")

All libraries loaded ✓


In [10]:
df = pd.read_excel(
    "../data/ML_Dataset_Aim3.xlsx",
    sheet_name="HFILOH Dwell time",
    header =3
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


Shape: (60, 27)
Columns: ['Row ID', 'Concentration\n(g/L)', 'Dwell Time\n(s)', 'Withdrawal\nSpeed (mm/s)', '–OH Count\n(cation)', 'F Chain\nLength', 'F Count\n(cation)', 'F Count\n(anion)', 'HBD\n(cation)', 'HBA\n(cation)', 'TPSA\n(cation, Å²)', 'Rot. Bonds\n(cation)', 'Ring Count\n(cation)', 'MW Cation\n(g/mol)', 'MW Anion\n(g/mol)', 'Bonded\nThickness (nm)', 'Bonded\nThickness_SD', 'Mobile\nThickness (nm)', 'Mobile\nThickness_SD', 'Total\nThickness (nm)', 'Total\nThickness_SD', 'Bonded\nFraction', 'WCA (°)', 'Unnamed: 23', 'HCA (°)', 'Unnamed: 25', 'Notes / data source']


,Row ID,Concentration\n(g/L),Dwell Time\n(s),Withdrawal\nSpeed (mm/s),–OH Count\n(cation),F Chain\nLength,F Count\n(cation),F Count\n(anion),HBD\n(cation),HBA\n(cation),...,Mobile\nThickness (nm),Mobile\nThickness_SD,Total\nThickness (nm),Total\nThickness_SD,Bonded\nFraction,WCA (°),Unnamed: 23,HCA (°),Unnamed: 25,Notes / data source
0,HFILOH-DS-001,0.5,30.0,1.0,1.0,6.0,13.0,18.0,1.0,3.0,...,0.633333,0.018257,0.823333,0.015275,0.230769,NaN,NaN,NaN,NaN,NaN
1,HFILOH-DS-002,0.5,60.0,1.0,1.0,6.0,13.0,18.0,1.0,3.0,...,0.551111,0.030505,0.982222,0.025874,0.438914,NaN,NaN,NaN,NaN,NaN
2,HFILOH-DS-003,0.5,90.0,1.0,1.0,6.0,13.0,18.0,1.0,3.0,...,0.537778,0.035158,0.966667,0.018028,0.443678,NaN,NaN,NaN,NaN,NaN
3,HFILOH-DS-004,0.5,120.0,1.0,1.0,6.0,13.0,18.0,1.0,3.0,...,0.516667,0.058571,0.987778,0.032702,0.476940,NaN,NaN,NaN,NaN,NaN
4,HFILOH-DS-005,0.5,150.0,1.0,1.0,6.0,13.0,18.0,1.0,3.0,...,0.514444,0.026247,1.008889,0.013642,0.490088,NaN,NaN,NaN,NaN,NaN


In [20]:
targets = [
    "Bonded\nThickness (nm)",
    "Mobile\nThickness (nm)", 
    "Bonded\nFraction",
    "WCA (°)",
    "HCA (°)"
]

print("Data coverage per target:")
for t in targets:
    if t in df.columns:
        n = df[t].notna().sum()
        print(f"  {t:30s}  {n}/{len(df)} rows filled")
    else:
        print(f"  {t:30s}  column not found")

Data coverage per target:
  Bonded
Thickness (nm)           41/60 rows filled
  Mobile
Thickness (nm)           41/60 rows filled
  Bonded
Fraction                 41/60 rows filled
  WCA (°)                         0/60 rows filled
  HCA (°)                         0/60 rows filled
